# RLFT experiment tracking on GEAP

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. It is the **RLFT counterpart** of
[`08_experiment_tracking.ipynb`](08_experiment_tracking.ipynb) (SFT). See
[`docs/notes/experiment-tracking.md`](../docs/notes/experiment-tracking.md)
for the verified SDK surface.

Experiment tracking has **two independent layers**:

- **Layer 1 — automatic.** Every tuning job streams train/validation curves to
  the Cloud console (**Agent Platform Studio → Tune and Distill → Monitor**)
  with no code, and they are **not** SDK-fetchable.
- **Layer 2 — opt-in (this notebook).** Log your **own** params + offline
  metrics to **Vertex AI Experiments**, via the `geap_tuning.experiments`
  helper. RLFT has no gold completion, so the metric **values** come from our
  own per-checkpoint offline eval (reward-scored answer accuracy), never from
  Layer 1. Summary metrics are **free**; per-step time-series curves also need a
  **Managed TensorBoard** (cost + ~10–20 min provisioning), so it is opt-in.

To keep cost down we reuse a **single**, cheap `gemini-2.5-flash`
RLFT-with-checkpoints job (declarative string-match reward), evaluate each
checkpoint, and log one Experiments run per checkpoint.

> **Requires live GCP and incurs tuning cost.** Have a real `.env` and
> `gcloud auth` in place. Tuning is **regional-only** (the `global` endpoint
> excludes tuning); keep the tuning/Experiments region aligned (`us-central1`).

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"  # cheap; a few epochs -> a few checkpoints to compare
EPOCHS = 3
ADAPTER_SIZE = 16
SAMPLES_PER_PROMPT = 4
DISPLAY_NAME = "geap-rlft-exp-tracking"
EXPERIMENT_NAME = "geap-rlft-checkpoint-eval"

cfg = load_config()
client = genai_client(cfg)  # regional client: tuning is excluded from the global endpoint
cfg

## 1. Build and stage the verifiable-math dataset

Deterministic `MATH_PROBLEMS` splits; each record carries a
`references={"ground_truth_answer": ...}` (no gold model turn).

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.rlft.data import build_rlft_dataset

paths = build_rlft_dataset("../datasets/rlft_math")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/experiment_tracking_rlft/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/experiment_tracking_rlft/val.jsonl")
train_uri, val_uri

## 2. Preflight the reward before spending money

RLFT auto-stops if >80% of reward calls fail, so we validate the declarative
string-match reward on one example first. This same `reward` object is reused for
the tune call.

In [ ]:
from geap_tuning.rlft.data import MATH_PROBLEMS, build_rlft_records, split_dataset
from geap_tuning.rlft.tune import build_string_match_reward_config, validate_reward_config

reward = build_string_match_reward_config()
train_records = build_rlft_records(split_dataset(MATH_PROBLEMS)[0])
validate_reward_config(
    client,
    project=cfg.project,
    location=cfg.location,
    sample_answer="Answer: 4",
    example_record=train_records[0],
    reward_config=reward,
)

## 3. Tune once, keeping every checkpoint

Reuse an existing job with the same display name if one exists (cost control).
`export_last_checkpoint_only=False` keeps per-checkpoint endpoints for the curve.

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, wait_for_tuning_job
from geap_tuning.rlft.tune import launch_rlft_job

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_rlft_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        epochs=EPOCHS,
        adapter_size=ADAPTER_SIZE,
        samples_per_prompt=SAMPLES_PER_PROMPT,
        reward_config=reward,
        export_last_checkpoint_only=False,
    )
job = wait_for_tuning_job(client, job.name)
job.state

## 4. Point Vertex AI Experiments at the tuning region

`init_experiment` creates/selects the experiment context so later `track_run`
calls attach to it. We pass no TensorBoard here — summary metrics don't need one.

In [ ]:
from geap_tuning.experiments import init_experiment

init_experiment(EXPERIMENT_NAME, project=cfg.project, location=cfg.location)

## 5. Evaluate every checkpoint on the held-out test split

Each checkpoint has its own endpoint, so we reward-score them independently and
collect answer accuracy per checkpoint.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import checkpoint_endpoint, list_checkpoints
from geap_tuning.rlft.evaluate import run_rlft_eval

_, _, test_problems = split_dataset(MATH_PROBLEMS)
test_records = build_rlft_records(test_problems)

results = []
for cp in list_checkpoints(job):
    endpoint = checkpoint_endpoint(job, cp.checkpoint_id)
    metrics = run_rlft_eval(
        test_records,
        generate_fn=lambda user_text, e=endpoint: generate(client, e, user_text),
    )
    results.append((cp, metrics))
    print(f"checkpoint {cp.checkpoint_id} (epoch {cp.epoch}): acc={metrics['accuracy']:.3f}")

## 6. Log one summary run per checkpoint

`track_run` opens a run and logs its params; `log_summary_metrics` records one
value per key. Params must be **scalar**, so we log the reward `"string_match"`
label, never the reward object. No TensorBoard required.

In [ ]:
from geap_tuning.experiments import log_summary_metrics, track_run

for cp, metrics in results:
    params = {
        "base_model": BASE_MODEL,
        "epochs": EPOCHS,
        "adapter_size": ADAPTER_SIZE,
        "samples_per_prompt": SAMPLES_PER_PROMPT,
        "reward": "string_match",
        "checkpoint_id": cp.checkpoint_id,
        "epoch": cp.epoch,
        "step": cp.step,
    }
    with track_run(f"{DISPLAY_NAME}-cp-{cp.checkpoint_id}", params=params):
        log_summary_metrics({"accuracy": metrics["accuracy"]})

## 7. (Optional) Managed TensorBoard time-series

> **Incurs cost and ~10–20 min provisioning.** Only run this cell if you want
> per-step curves in a user-owned TensorBoard.

Time-series metrics live in a Managed TensorBoard, so we provision/attach one
(reused by display name, shared with the SFT demo) and re-init the experiment
with it before logging the accuracy-vs-epoch curve as a single run.

In [ ]:
from geap_tuning.experiments import get_or_create_tensorboard, log_timeseries_metrics

tensorboard = get_or_create_tensorboard(
    "geap-tuning-tb", project=cfg.project, location=cfg.location
)
init_experiment(
    EXPERIMENT_NAME, project=cfg.project, location=cfg.location, tensorboard=tensorboard
)
with track_run(f"{DISPLAY_NAME}-curve"):
    for cp, metrics in sorted(results, key=lambda r: r[0].epoch):
        log_timeseries_metrics({"accuracy": metrics["accuracy"]}, step=cp.epoch)
tensorboard

## 8. Compare runs

`experiment_dataframe` returns a pandas table of every run's params + summary
metrics — the same data you see under **Agent Platform Studio → Experiments**.

In [ ]:
from geap_tuning.experiments import experiment_dataframe

experiment_dataframe(EXPERIMENT_NAME)

## Next steps

This mirrors the SFT tracking demo
([`08_experiment_tracking.ipynb`](08_experiment_tracking.ipynb)) for RLFT. For
the automatic metric keys per method and the full API surface see
[`docs/notes/experiment-tracking.md`](../docs/notes/experiment-tracking.md).